# 🎼 Orchestrator-Worker with the `Send` API

## Learning Objectives
In this notebook, you will learn:
1. **The pattern** - What makes orchestrator-worker different from parallelization, and why it is also called map-reduce
2. **A runtime plan** - Have an orchestrator LLM return a variable-length list of subtasks with structured output
3. **Dynamic dispatch with `Send`** - Spawn exactly one worker per planned subtask and merge their results with a reducer
4. **Proving the dynamism** - Show that the worker count follows the input, and watch workers finish in their own order
5. **The anti-pattern** - Recognise a "plan" that never drives execution because the workers are fixed nodes

## Where this fits
Module 07 ran a **fixed** set of branches in parallel. Here an LLM decides at run time how many branches to run and what each one does. `02_orchestrator_worker_lcel_map.ipynb` then rebuilds the same report generator in LCEL with `Runnable.map()`.

## Prerequisites
- Fan-out/fan-in and reducers — `modules/07_parallelization/notebooks/01_parallelization_langgraph.ipynb`
- `with_structured_output` — `modules/04_langchain_agents_and_middleware/notebooks/03_structured_output.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env` (no other keys needed)

---
## 📖 Part 1: The Orchestrator-Worker Pattern

The Orchestrator-Worker pattern uses a central LLM (the **orchestrator**) to analyze an incoming request, dynamically break it into an unknown-in-advance number of subtasks, and dispatch each subtask to a **worker** LLM. A final step **synthesizes** all worker outputs into one cohesive result.

The single feature that makes this pattern what it is: **the orchestrator decides the number and shape of subtasks at runtime, based on the specific input** — not at graph-build time.

| Property | Value |
|---|---|
| Origin | Anthropic, *Building Effective Agents* (Dec 2024) — [anthropic.com/research/building-effective-agents](https://www.anthropic.com/research/building-effective-agents), "Orchestrator-workers" section |
| Mechanism | LangGraph `Send` API — [langchain-ai.github.io/langgraph](https://langchain-ai.github.io/langgraph/how-tos/graph-api/#map-reduce-and-the-send-api) |

### 1.1 ⚖️ Orchestrator-Worker vs. Parallelization

It's easy to confuse this with the Parallelization pattern (module 07, `modules/07_parallelization/notebooks/02_parallelization_sectioning_and_voting.ipynb`), because both run multiple LLM calls concurrently and both end with a synthesis step. The difference is in *how* the parallel branches are decided:

| | Parallelization | Orchestrator-Worker |
|---|---|---|
| Who decides the subtasks | The developer, at graph-build time | The orchestrator LLM, at runtime |
| Number of branches | Fixed (e.g. always 3 nodes) | Variable — depends on the input |
| Branch identity | Known in advance (e.g. "characters", "setting", "premise") | Generated dynamically (e.g. however many report sections the topic needs) |
| Graph shape | Static edges, hardcoded in `add_edge` | Dynamic fan-out via `Send`, computed inside a conditional edge |

### 1.2 🗺️ The Same Shape Is Called Map-Reduce

You will often see this graph described as **map-reduce**. The vocabulary maps one to one: the orchestrator's plan is the list to map over, the `Send` fan-out is the map, and the synthesizer is the reduce.

| Phase | Description | Example |
|-------|-------------|---------|
| **Map** | Break a task into smaller sub-tasks, processing each sub-task in parallel | Generate multiple questions, then answer each one in parallel |
| **Reduce** | Aggregate the results across all completed, parallelized sub-tasks | Combine all Q&A pairs into a comprehensive report |

The reduce step can be plain code (this notebook joins the sections) or another LLM call that rewrites the collected results into one answer.

> **Key Insight**: `Send` is the mechanism, not the pattern. The pattern is "an LLM decides the subtasks at run time"; `Send` is how LangGraph spawns a number of workers nobody wrote down in advance.

---
## 📦 Step 1: Environment Setup

Load the API key, then create the chat models through the course factory.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: One model for planning, one for writing sections
# ============================================================================
from aied_helpers import get_openai_llm

# They can be the same underlying model - kept as two variables for clarity.
orchestrator_llm = get_openai_llm(model="gpt-4o-mini")
worker_llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {orchestrator_llm.model_name}")

---
## 🧭 Part 2: The Orchestrator's Runtime Plan

The orchestrator doesn't just describe a plan in prose - it returns a **structured, variable-length list of sections**. That list is what actually drives how many workers get spawned.

The system prompt matters as much as the schema. An earlier version said "simple topics might need 2 sections, broad topics 5+", and `gpt-4o-mini` answered with five sections — Introduction and Conclusion included — for *every* topic. Asking for one section per distinct question, with no padding sections, is what lets the plan size follow the input.

In [ ]:
# ============================================================================
# PLAN STRUCTURE: What the orchestrator produces
# ============================================================================
from pydantic import BaseModel, Field

class Section(BaseModel):
    """A single report section identified by the orchestrator."""
    name: str = Field(description="Short title of this report section")
    description: str = Field(
        description="What this section should cover - scope, angle, key points to hit"
    )

class ReportPlan(BaseModel):
    """The orchestrator's full breakdown of a topic into sections."""
    sections: list[Section] = Field(
        description="The sections needed to cover this topic well - can be as few or as many as the topic warrants"
    )

# Force the orchestrator's output into our ReportPlan schema
planner = orchestrator_llm.with_structured_output(ReportPlan)

print(f"✅ Planner ready: returns {ReportPlan.__name__} with fields {list(ReportPlan.model_fields)}")

### 2.1 🧠 State and the Orchestrator Node

This is the planning step: given only a topic, the orchestrator decides how many sections are needed and what each should cover. It is an ordinary node: it reads the topic from state and writes the plan back.

The graph needs two state schemas:

- **`ReportState`**: the graph-level state, visible to the orchestrator and synthesizer
- **`WorkerState`**: the narrow slice of state each individual worker actually needs

`completed_sections` uses the `operator.add` reducer so that when an *unknown number* of parallel workers each return one section, LangGraph appends them all into a single list instead of overwriting each other.

In [ ]:
# ============================================================================
# STATE + ORCHESTRATOR: Graph-level state and the planning node
# ============================================================================
import operator
from typing import Annotated

from langchain_core.messages import HumanMessage, SystemMessage
from typing_extensions import TypedDict


class ReportState(TypedDict):
    topic: str                                         # Input topic for the report
    sections: list[Section]                            # Orchestrator's dynamic plan
    completed_sections: Annotated[list, operator.add]  # Fan-in target for worker outputs
    final_report: str                                  # Synthesized result


class WorkerState(TypedDict):
    topic: str        # Needed for context so each section stays coherent with the whole
    section: Section  # The single section this worker is responsible for


PLANNER_SYSTEM_PROMPT = (
    "You are a report planning orchestrator. Given a topic, plan the sections "
    "of a short report. Create one section for each distinct question the topic "
    "actually asks. A narrow topic with a single factual answer needs very few "
    "sections. A broad topic that spans several fields gets a separate section "
    "for each field and each major sub-question. Never add Introduction, "
    "Conclusion, Summary or other padding sections."
)


def orchestrate_report(state: ReportState):
    """Analyze the topic and dynamically decide how many sections it needs."""
    plan = planner.invoke(
        [
            SystemMessage(content=PLANNER_SYSTEM_PROMPT),
            HumanMessage(content=f"Plan the sections for a report on: {state['topic']}"),
        ]
    )
    print(f"🗺️  Orchestrator planned {len(plan.sections)} section(s): {[s.name for s in plan.sections]}")
    return {"sections": plan.sections}


print("✅ State schemas and orchestrator defined")

---
## 🚀 Part 3: Dynamic Dispatch with `Send`

This is the crux of the pattern. A regular conditional edge returns the name of *one* next node. `Send` lets a conditional edge return a **list of dispatch instructions** - one `Send("llm_call", {...})` per section the orchestrator decided on. LangGraph spawns exactly that many parallel invocations of the `llm_call` node, each with its own isolated `WorkerState`.

Because `state["sections"]` came from the orchestrator's LLM call, **the number of `Send` objects generated here is only known at runtime** - this is what a fixed `add_edge` fan-out (as in the Parallelization pattern) cannot express.

In [ ]:
# ============================================================================
# DYNAMIC DISPATCH: One Send() per orchestrator-planned section
# ============================================================================
from langgraph.types import Send


def assign_workers(state: ReportState):
    """Fan out to one worker per section - the count is decided by the orchestrator, not us."""
    return [
        Send("llm_call", {"topic": state["topic"], "section": section})
        for section in state["sections"]
    ]


print("✅ assign_workers returns one Send per planned section")

### 3.1 👷 The Worker Node

Each worker only ever sees one `Section` at a time (its own slice of `WorkerState`). Critically, the worker actually uses the orchestrator's `section.description` to write its content, so the plan is load-bearing rather than decorative (Part 5 shows what happens when it is not).

In [ ]:
# ============================================================================
# WORKER: Writes one section, driven by the orchestrator's plan
# ============================================================================

def llm_call(state: WorkerState):
    """Write one report section based on the orchestrator's description of it."""
    section = state["section"]

    section_prompt = (
        f"Write the '{section.name}' section of a report on: {state['topic']}\n\n"
        f"This section should specifically cover: {section.description}\n\n"
        f"Write 2-4 focused paragraphs of body text only. Do not include a title or "
        f"heading line (one will be added separately) and do not repeat content that "
        f"belongs in other sections."
    )

    response = worker_llm.invoke(section_prompt)
    formatted_section = f"## {section.name}\n\n{response.content}"

    # Returned as a single-item list so the operator.add reducer appends it
    # alongside every other worker's output, however many there turn out to be.
    return {"completed_sections": [formatted_section]}


print("✅ Worker defined — it writes ONE section per call")

### 3.2 🧵 The Synthesizer (the Reduce Step)

Once every dynamically-spawned worker has finished, the synthesizer combines all completed sections into the final report.

In [ ]:
# ============================================================================
# SYNTHESIZER: Combine all worker outputs
# ============================================================================

def synthesize_report(state: ReportState):
    """Combine every completed section into one final report."""
    combined = "\n\n".join(state["completed_sections"])
    return {"final_report": f"# Report: {state['topic']}\n\n{combined}"}


print("✅ Synthesizer defined")

### 3.3 🏗️ Building the Graph

Notice `add_conditional_edges` is what wires in the dynamic fan-out - `llm_call` is registered as a possible destination, but *how many times* it runs per invocation is decided by `assign_workers` at runtime, not by the graph definition.

In [ ]:
# ============================================================================
# GRAPH: Orchestrator -> dynamic workers -> synthesizer
# ============================================================================
from langgraph.graph import END, START, StateGraph

orchestrator_worker_builder = StateGraph(ReportState)

orchestrator_worker_builder.add_node("orchestrator", orchestrate_report)
orchestrator_worker_builder.add_node("llm_call", llm_call)
orchestrator_worker_builder.add_node("synthesizer", synthesize_report)

orchestrator_worker_builder.add_edge(START, "orchestrator")

# The dynamic fan-out: assign_workers returns a variable-length list of Send()
# objects, one per orchestrator-planned section.
orchestrator_worker_builder.add_conditional_edges(
    "orchestrator", assign_workers, ["llm_call"]
)

orchestrator_worker_builder.add_edge("llm_call", "synthesizer")
orchestrator_worker_builder.add_edge("synthesizer", END)

report_workflow = orchestrator_worker_builder.compile()

print("✅ Graph compiled")

In [ ]:
# ============================================================================
# INSPECT THE GRAPH: Print nodes and edges (no network call needed)
# ============================================================================
g = report_workflow.get_graph()

print("🧩 Nodes:", [n for n in g.nodes if not n.startswith("__")])
print("🔗 Edges:")
for e in g.edges:
    kind = "conditional" if e.conditional else "fixed"
    print(f"   {e.source:>18} ──▶ {e.target:<18} ({kind})")

The `orchestrator ──▶ llm_call` edge is the only conditional one, and it appears **once**, however many workers a run spawns. The graph drawing cannot show the fan-out count because the graph does not know it.

---
## 🧪 Part 4: Proving the Dynamism

We run the same graph on two topics of deliberately different scope. If this were parallelization, the worker count would be identical every run; here it should follow the topic.

We use `stream(stream_mode=["updates", "values"])` instead of `invoke`: `"updates"` prints each worker's return value the moment it finishes, and `"values"` gives the full state after each step, with the reducer applied. Two orders are worth comparing: the order workers **finish** (the stream) and the order sections appear in the **final state** (the reducer).

In [ ]:
# ============================================================================
# TEST: Worker count should follow the topic, not the code
# ============================================================================
import time

test_topics = [
    "What is the boiling point of water at sea level?",                  # narrow
    "A buyer's guide to electric cars: purchase cost, range, charging, "
    "maintenance, resale value and government incentives",               # broad
]

results = {}
for topic in test_topics:
    print(f"\n{'=' * 70}\nTOPIC: {topic}\n{'=' * 70}")

    start = time.perf_counter()
    finish_order = []
    final_state = {}
    # "updates" = what each node just returned (as it finishes);
    # "values"  = the full state after each step, with reducers applied.
    for mode, chunk in report_workflow.stream({"topic": topic}, stream_mode=["updates", "values"]):
        if mode == "values":
            final_state = chunk
            continue
        for node, delta in chunk.items():
            if node == "llm_call":
                name = delta["completed_sections"][0].splitlines()[0].removeprefix("## ")
                finish_order.append(name)
                print(f"   👷 worker finished at {time.perf_counter() - start:5.1f}s: {name}")
    elapsed = time.perf_counter() - start

    planned = [s.name for s in final_state["sections"]]
    merged = [s.splitlines()[0].removeprefix("## ") for s in final_state["completed_sections"]]
    results[topic] = len(planned)
    print(f"\n⏱️  Wall time: {elapsed:.1f}s")
    print(f"👷 Workers spawned: {len(finish_order)} (planned sections: {len(planned)})")
    print(f"🏁 Finish order matches plan order: {finish_order == planned}")
    print(f"📋 Reducer (final state) order matches plan order: {merged == planned}")
    print(f"\nFinal report preview:\n{final_state['final_report'][:400]}...")

In [ ]:
# ============================================================================
# CHECK: Did the two topics really spawn different numbers of workers?
# ============================================================================
for topic, n in results.items():
    print(f"{n:>2} workers  ←  {topic[:60]}")

counts = set(results.values())
print("\n✅ Worker count changed with the input" if len(counts) > 1 else "\n⚠️ Same count for both topics — rerun or try a broader topic")

Read the two order checks. The **stream** reports workers as they finish, so a short section can overtake a long one. The **reducer**, however, merges the parallel writes in the order the `Send` objects were created — the plan order — so the final report reads in the planned order no matter which worker finished first. (On a given run the finish order may happen to match; the reducer order always does.)

---
## 🚫 Part 5: The Anti-Pattern — A Plan Nobody Reads

A common broken version of this pattern keeps the orchestrator but wires a **fixed** set of worker nodes with `add_edge`. The workers read the original request directly and never consume the plan. The plan is generated, printed and then ignored — this is Parallelization (module 07) with a decorative LLM call in front.

To show it without extra LLM cost, the fixed workers below are stubs: the point is which workers run, not what they write. The orchestrator is the real planner from Part 2.

> **Diagnosis**: if the list of workers that ran is identical for every input while the plan changes, the plan does not drive execution.

In [ ]:
# ============================================================================
# ANTI-PATTERN: Five fixed workers that never look at the plan
# ============================================================================
class FixedState(TypedDict):
    topic: str
    sections: list[Section]                          # Planned... and then ignored
    workers_executed: Annotated[list, operator.add]


def make_fixed_worker(name: str):
    def worker(state: FixedState):
        # Reads only the topic; state["sections"] is never consulted.
        return {"workers_executed": [name]}
    return worker


fixed_builder = StateGraph(FixedState)
fixed_builder.add_node("orchestrator", orchestrate_report)
fixed_workers = [
    "background_worker", "analysis_worker", "examples_worker", "risks_worker", "outlook_worker",
]
for name in fixed_workers:
    fixed_builder.add_node(name, make_fixed_worker(name))
    fixed_builder.add_edge("orchestrator", name)   # static fan-out, decided when the code was written
    fixed_builder.add_edge(name, END)
fixed_builder.add_edge(START, "orchestrator")
fixed_workflow = fixed_builder.compile()

for topic in test_topics:
    result = fixed_workflow.invoke({"topic": topic})
    print(f"   planned {len(result['sections'])} sections, but workers executed: {len(result['workers_executed'])} → {sorted(result['workers_executed'])}\n")

The plan changes with the topic, but the workers executed are the same five every time — too many for the narrow topic, too few for the broad one. Compare with Part 4, where the same planner produced a different number of `llm_call` runs per topic. The fix is always the same: dispatch from the plan (`Send` per item) and make each worker consume its item.

---
## 📝 Summary

In this notebook, we learned:

### 1. The pattern
- The orchestrator LLM (here, `planner`) outputs a variable-length `ReportPlan`, and that plan - not a hardcoded set of node names - determines what runs next.
- The same graph is often called **map-reduce**: plan = the list, `Send` = map, synthesizer = reduce.

### 2. A runtime plan
- A Pydantic schema with a `list[Section]` field lets the LLM return a variable-length plan. The prompt must allow a small plan — padding instructions or numeric anchors make every plan the same size.

### 3. Dynamic dispatch with `Send`
- LangGraph's `Send` API is what lets a single conditional edge fan out to *N* worker executions, where *N* is only known once the orchestrator has run. Without `Send` (or an equivalent), you're stuck wiring a fixed set of nodes at graph-build time, which collapses back into Parallelization.
- `Annotated[list, operator.add]` collects one item per worker; the merged list follows `Send` order, while `stream_mode="updates"` shows workers in completion order.

### 4. Proving the dynamism
- The same compiled graph spawned a different number of workers for a narrow and a broad topic, with no code change.

### 5. The anti-pattern
- Each `llm_call` invocation uses `section.description` from the orchestrator's output to do its job. Fixed `add_edge` workers that ignore the plan are parallelization, whatever the graph is called.

### Next Steps
- **`02_orchestrator_worker_lcel_map.ipynb`**: the same report generator in LCEL with `Runnable.map()`, and what it gives up compared with `Send`